# 세션3 — Post-retrieval / Rerank

세션2에서는 검색기를 그대로 두고 **질문을 재작성**(Multi-Query, HyDE)해서 정답 문서의 순위를 끌어올렸습니다. 이번 세션에서는 **같은 문제, 같은 질문**을 놓고 반대 방향으로 접근합니다: 질문은 그대로 두고, 검색 후 회수된 후보들의 **순위를 다시 매기는(Rerank)** 방법입니다.

1. **Threshold 필터링** — 점수가 낮은 문서를 그냥 잘라내면 될까?
2. **Bi-Encoder Reranker** — 후보를 독립적으로 다시 임베딩해서 정렬하면 나아질까?
3. **Cross-Encoder Reranker** — 질문과 문서를 "함께" 보고 관련성을 판단하면?

세션2와 동일하게 Dense Retriever 하나만 기준으로 놓고 비교합니다(Hybrid까지 섞으면 BM25 때문에 순위 계산이 헷갈립니다). 저장소 루트의 키오스크 이용실태 조사 PDF와 vectorstore를 그대로 재사용합니다.

In [2]:
import os

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

# 세션1/2와 마찬가지로 저장소 루트의 공유 data/, vectorstore/를 직접 참조합니다.
DATA_PATH = os.path.join("..", "data", "키오스크(무인정보단말기) 이용실태 조사.pdf")
VECTORSTORE_DIR = os.path.join("..", "vectorstore")
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")

# 정답 문서: metadata상 page=4(PDF 하단 표기로는 5페이지) - "조사 배경 및 목적" 문단
# (세션2 노트북은 이 문단을 page=5로 표기했는데, 실제로 확인해보면 해당 문장은
#  page=4 청크에 있고 page=5 청크는 "무인정보단말기 접근성 지침 개정"이라는 다른 내용입니다.)
ANSWER_PAGE = 4

docs = PyPDFLoader(DATA_PATH).load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
splited_docs = splitter.split_documents(docs)
print(f"원본 페이지 수: {len(docs)}, 청크 수: {len(splited_docs)}")


def find_rank(results, target_page):
    """검색 결과에서 target_page 문서가 몇 번째(1-indexed)에 있는지 찾습니다. 없으면 None."""
    for i, doc in enumerate(results):
        if doc.metadata.get("page") == target_page:
            return i + 1
    return None


def show_results(docs, title):
    """검색 결과를 순위, 페이지 번호와 함께 짧게 출력합니다."""
    print(f"--- {title} ---")
    for i, doc in enumerate(docs):
        page = doc.metadata.get("page")
        preview = doc.page_content[:55].strip().replace(chr(10), " ")
        print(f"[{i + 1}순위][p{page}] {preview}...")
    print()


원본 페이지 수: 59, 청크 수: 73


## 세션1/2 복습 — Dense Retriever 다시 불러오기

이 노트북도 별도 파일이라 이전 세션의 retriever가 메모리에 없습니다. 세션2와 동일하게 Dense Retriever만 다시 불러옵니다.

In [3]:
from langchain_chroma import Chroma

vectorstore = Chroma(persist_directory=VECTORSTORE_DIR, embedding_function=embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 8})


## Before — Dense Retriever 후보 확인

보고서 5페이지(footer 표기 기준)에는 이런 내용이 있습니다.

> 비대면 소비 트렌드 확산에 따라 ... 키오스크 설치 확대는 인건비 절감‧비대면 서비스 선호층 증가에 따른 수요 충족 등의 긍정적 효과가 있으나 ...

세션2와 똑같은 질문 **"키오스크가 갑자기 왜 이렇게 흔해졌어?"**를 던져, Dense Retriever가 회수한 top-8 후보 안에서 정답 청크(`ANSWER_PAGE`)가 몇 위인지 확인합니다. 이번엔 질문을 재작성하지 않고, **이 8개 후보를 그대로 재정렬**하는 방법만 시도합니다.

In [4]:
query = "키오스크가 갑자기 왜 이렇게 흔해졌어?"

candidates = dense_retriever.invoke(query)
show_results(candidates, f"Dense 후보 (rerank 전): '{query}'")

before_rank = find_rank(candidates, ANSWER_PAGE)
print(f"정답 문서(p{ANSWER_PAGE}) 순위: {before_rank}위 / {len(candidates)}개")


--- Dense 후보 (rerank 전): '키오스크가 갑자기 왜 이렇게 흔해졌어?' ---
[1순위][p27] 키오스크(무인정보단말기)이용 실태조사 28 시장조사국 시장감시팀 □(중단 이유)키오스크 이용을 도중...
[2순위][p22] 키오스크(무인정보단말기)이용 실태조사 23 시장조사국 시장감시팀 □(이용 이유)키오스크를 이용하게...
[3순위][p36] 키오스크(무인정보단말기)이용 실태조사 37 시장조사국 시장감시팀 -60대 이상은 다른 연령대에 비해...
[4순위][p4] 키오스크(무인정보단말기)이용 실태조사 5 시장조사국 시장감시팀 Ⅰ조사 개요1. 조사 배경 및 목적□...
[5순위][p51] 키오스크(무인정보단말기)이용 실태조사 52 시장조사국 시장감시팀 ㅇ 그러나 「장차법」시행일인’23년...
[6순위][p11] 키오스크(무인정보단말기)이용 실태조사 12 시장조사국 시장감시팀 [표2-2-3] 청년층·비장애인의...
[7순위][p34] 키오스크(무인정보단말기)이용 실태조사 35 시장조사국 시장감시팀 ㅇ (③큰 화면‧ 글씨,음성 안내...
[8순위][p6] 키오스크(무인정보단말기)이용 실태조사 7 시장조사국 시장감시팀 Ⅱ일반 현황1. 키오스크(무인정보단말...

정답 문서(p4) 순위: 4위 / 8개


정답 청크가 8개 중 4위입니다. 완전히 놓친 건 아니지만, 1~3위 자리를 "키오스크 이용 중단 이유", "이용 이유", "연령대별 교육 참여"처럼 **질문과 표면적으로만 비슷한 다른 문단들**이 차지하고 있습니다. 이 후보 묶음을 세 가지 방법으로 다시 정렬해보겠습니다.

## ① Threshold 필터링 — 점수로 자르면 될까?

가장 단순한 post-retrieval 방법은 "유사도 점수가 일정 기준(threshold) 이하인 문서는 버린다"입니다. `similarity_search_with_relevance_scores`로 Dense 점수를 직접 확인해봅니다.

In [5]:
scored_docs = vectorstore.similarity_search_with_relevance_scores(query, k=8)

print(f"--- Dense 유사도 점수: '{query}' ---")
for i, (doc, score) in enumerate(scored_docs):
    page = doc.metadata.get("page")
    preview = doc.page_content[:50].strip().replace(chr(10), " ")
    print(f"[{i + 1}순위][score={score:.3f}][p{page}] {preview}...")


--- Dense 유사도 점수: '키오스크가 갑자기 왜 이렇게 흔해졌어?' ---
[1순위][score=0.317][p27] 키오스크(무인정보단말기)이용 실태조사 28 시장조사국 시장감시팀 □(중단 이유)키오스크 이...
[2순위][score=0.311][p22] 키오스크(무인정보단말기)이용 실태조사 23 시장조사국 시장감시팀 □(이용 이유)키오스크를...
[3순위][score=0.295][p36] 키오스크(무인정보단말기)이용 실태조사 37 시장조사국 시장감시팀 -60대 이상은 다른 연령...
[4순위][score=0.289][p4] 키오스크(무인정보단말기)이용 실태조사 5 시장조사국 시장감시팀 Ⅰ조사 개요1. 조사 배경...
[5순위][score=0.275][p51] 키오스크(무인정보단말기)이용 실태조사 52 시장조사국 시장감시팀 ㅇ 그러나 「장차법」시행일...
[6순위][score=0.273][p11] 키오스크(무인정보단말기)이용 실태조사 12 시장조사국 시장감시팀 [표2-2-3] 청년층·비...
[7순위][score=0.272][p34] 키오스크(무인정보단말기)이용 실태조사 35 시장조사국 시장감시팀 ㅇ (③큰 화면‧ 글씨,음...
[8순위][score=0.256][p6] 키오스크(무인정보단말기)이용 실태조사 7 시장조사국 시장감시팀 Ⅱ일반 현황1. 키오스크(무...


점수만 보면 문제가 보입니다. 1위(0.317)와 정답 문서인 4위(0.289)의 점수 차이가 **0.028점**밖에 안 나고, 8개 전체가 0.256~0.317 사이의 좁은 범위에 몰려 있습니다.

- 기준을 0.29 근처로 잡으면 → 정답 문서는 겨우 살아남지만, 그보다 점수가 높은 **관련 없는 문서 3개**는 그대로 남습니다.
- 기준을 0.32 이상으로 높이면 → 관련 없는 문서까지 다 걸러지지만 **정답 문서도 같이 잘립니다.**

점수는 "질문과 문서가 표면적으로 얼마나 가까운가"만 볼 뿐, "이 문서가 질문에 실제로 답하는가"는 보지 못합니다. 그래서 점수 하나로 자르는 방식(threshold)만으로는 순위 문제를 해결할 수 없습니다.

## ② Bi-Encoder Reranker — 후보를 독립적으로 다시 임베딩하면?

Threshold보다 한 단계 나은 방법은 후보들을 **다시 정렬**하는 것입니다. 가장 쉬운 재정렬 방법은 질문과 각 문서를 **각각 독립적으로 임베딩**한 뒤 코사인 유사도로 순위를 다시 매기는 것 — 이게 Bi-Encoder Reranker입니다. (Dense Retriever와 원리는 완전히 동일하고, "후보 위에서 한 번 더" 적용한다는 점만 다릅니다.)

In [6]:
import numpy as np


def cosine_similarity(vec_a, vec_b):
    vec_a, vec_b = np.array(vec_a), np.array(vec_b)
    return float(np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b)))


def bi_encoder_rerank(query, docs):
    """질문과 문서를 각각 독립적으로 임베딩해 코사인 유사도로 재정렬합니다."""
    query_vec = embeddings.embed_query(query)
    scored = []
    for doc in docs:
        doc_vec = embeddings.embed_query(doc.page_content)
        scored.append((cosine_similarity(query_vec, doc_vec), doc))
    scored.sort(key=lambda x: -x[0])
    return scored


bi_encoder_reranked = bi_encoder_rerank(query, candidates)

print(f"--- Bi-Encoder Reranker 결과: '{query}' ---")
for i, (score, doc) in enumerate(bi_encoder_reranked):
    page = doc.metadata.get("page")
    preview = doc.page_content[:50].strip().replace(chr(10), " ")
    print(f"[{i + 1}순위][sim={score:.4f}][p{page}] {preview}...")

after_rank_bi = find_rank([doc for _, doc in bi_encoder_reranked], ANSWER_PAGE)
print(f"\n정답 문서(p{ANSWER_PAGE}) 순위: {before_rank}위 → {after_rank_bi}위")


--- Bi-Encoder Reranker 결과: '키오스크가 갑자기 왜 이렇게 흔해졌어?' ---
[1순위][sim=0.5170][p27] 키오스크(무인정보단말기)이용 실태조사 28 시장조사국 시장감시팀 □(중단 이유)키오스크 이...
[2순위][sim=0.5132][p22] 키오스크(무인정보단말기)이용 실태조사 23 시장조사국 시장감시팀 □(이용 이유)키오스크를...
[3순위][sim=0.5012][p36] 키오스크(무인정보단말기)이용 실태조사 37 시장조사국 시장감시팀 -60대 이상은 다른 연령...
[4순위][sim=0.4971][p4] 키오스크(무인정보단말기)이용 실태조사 5 시장조사국 시장감시팀 Ⅰ조사 개요1. 조사 배경...
[5순위][sim=0.4876][p51] 키오스크(무인정보단말기)이용 실태조사 52 시장조사국 시장감시팀 ㅇ 그러나 「장차법」시행일...
[6순위][sim=0.4859][p11] 키오스크(무인정보단말기)이용 실태조사 12 시장조사국 시장감시팀 [표2-2-3] 청년층·비...
[7순위][sim=0.4852][p34] 키오스크(무인정보단말기)이용 실태조사 35 시장조사국 시장감시팀 ㅇ (③큰 화면‧ 글씨,음...
[8순위][sim=0.4741][p6] 키오스크(무인정보단말기)이용 실태조사 7 시장조사국 시장감시팀 Ⅱ일반 현황1. 키오스크(무...

정답 문서(p4) 순위: 4위 → 4위


순위가 **그대로 4위**입니다. 심지어 각 순위 사이의 점수 차이(0.517 → 0.497)도 Threshold에서 본 것과 비슷하게 근소합니다.

이유는 Bi-Encoder Reranker가 결국 Dense Retriever와 **같은 원리**이기 때문입니다. 질문과 문서를 따로따로 벡터로 만들고 거리만 재는 방식이라, Dense가 이미 놓친(또는 애매하게 판단한) 것을 다시 계산해도 **새로운 정보가 추가되지 않습니다.** "다시 임베딩해서 정렬"만으로는 한계가 있다는 뜻입니다.

## ③ Cross-Encoder Reranker — 질문과 문서를 "함께" 보면?

Bi-Encoder는 질문과 문서를 따로 인코딩하기 때문에 둘 사이의 **상호작용**을 볼 수 없습니다. Cross-Encoder Reranker는 **질문과 문서를 한 쌍으로 묶어서 동시에** 모델에 입력하고, "이 쌍이 얼마나 관련 있는지" 점수 하나를 직접 계산합니다. 후보 하나하나를 질문과 짝지어 정밀하게 채점하는 대신, 후보 수만큼 연산이 필요해 더 느립니다.

`langchain_classic`의 `CrossEncoderReranker`와 `BAAI/bge-reranker-v2-m3` 모델을 사용합니다.

In [7]:
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
top_n = 4
reranker = CrossEncoderReranker(model=cross_encoder, top_n=top_n)

cross_encoder_reranked = reranker.compress_documents(candidates, query)

show_results(cross_encoder_reranked, f"Cross-Encoder rerank 이후 (top {top_n}개)")

after_rank_ce = find_rank(cross_encoder_reranked, ANSWER_PAGE)
print(f"정답 문서(p{ANSWER_PAGE}) 순위: {before_rank}위 → {after_rank_ce}위")


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

--- Cross-Encoder rerank 이후 (top 4개) ---
[1순위][p4] 키오스크(무인정보단말기)이용 실태조사 5 시장조사국 시장감시팀 Ⅰ조사 개요1. 조사 배경 및 목적□...
[2순위][p22] 키오스크(무인정보단말기)이용 실태조사 23 시장조사국 시장감시팀 □(이용 이유)키오스크를 이용하게...
[3순위][p6] 키오스크(무인정보단말기)이용 실태조사 7 시장조사국 시장감시팀 Ⅱ일반 현황1. 키오스크(무인정보단말...
[4순위][p51] 키오스크(무인정보단말기)이용 실태조사 52 시장조사국 시장감시팀 ㅇ 그러나 「장차법」시행일인’23년...

정답 문서(p4) 순위: 4위 → 1위


정답 문서가 **1위로 승격**합니다. 원점수를 직접 찍어보면 차이가 더 뚜렷합니다 — 정답 문서는 0.71점, 2위는 0.26점으로 **거의 3배** 차이가 납니다. Bi-Encoder에서 순위 사이 점수 차이가 근소했던 것과 대조적으로, Cross-Encoder는 "이 문단이 진짜 답이다"를 확신을 갖고 골라냅니다.

세 방식의 정답 문서 순위를 정리하면 이렇게 달라집니다.

| 방식 | 정답 문서 순위 | 1위와의 점수차 |
|---|---|---|
| Dense (재정렬 전) | 4위 / 8개 | 0.028 (근소) |
| Bi-Encoder Reranker | 4위 / 8개 (개선 없음) | 0.020 (근소) |
| **Cross-Encoder Reranker** | **1위** | **0.45 (압도적)** |

Cross-Encoder만 질문("왜 흔해졌어?")과 각 후보 문서를 함께 보고, "설치 확대 배경을 실제로 서술한 문단"과 "키오스크 이용 관련 언급만 있는 다른 문단들"을 확실히 구분해냅니다.

## 한계 — Reranker가 만능은 아닙니다

Cross-Encoder Reranker는 강력하지만 두 가지 한계가 있습니다.

1. **1차 후보에 없으면 애초에 살릴 수 없습니다.** Cross-Encoder는 `dense_retriever`가 회수한 8개 후보 안에서만 순위를 다시 매깁니다. 만약 1차 검색 단계에서 정답 청크가 후보에 아예 안 들어왔다면(k를 더 작게 잡았거나, 질문이 더 어긋났다면), 아무리 좋은 reranker를 써도 찾을 수 없습니다. 즉 **recall(놓치지 않고 찾아내는 정도)은 여전히 1차 retriever의 몫**이고, reranker는 그 안에서 **precision(순위)**만 개선합니다. 이번 세션3의 "후보 재정렬"과 세션2의 "질문 재작성(Multi-Query/HyDE)"은 서로 다른 지점(순위 vs recall)을 보완하는 관계입니다.
2. **후보 수만큼 비용이 듭니다.** 질문-문서 쌍마다 모델을 한 번씩 통과시켜야 하므로, 후보가 많아질수록 느려지고 비용도 커집니다.

## 실전 패턴 — "넉넉히 찾고, 정확히 추리기"

실무에서는 `top_n`개만 처음부터 검색하지 않고, **`top_n`보다 훨씬 넉넉한 후보를 1차로 확보한 뒤 Cross-Encoder로 추려서 `top_n`개만 남기는** 패턴을 씁니다. 위에서는 k=8 고정으로 봤지만, 이번엔 `top_n * 5`개를 1차로 폭넓게 검색해 정답이 후보군에 들어올 확률 자체를 높이고, 그 다음 Cross-Encoder Reranker로 최종 `top_n`개를 골라내는 하나의 체인으로 구성해봅니다.

In [ ]:
from langchain_core.runnables import RunnableLambda

# 1차 검색은 top_n의 5배만큼 넉넉히 후보를 확보합니다.
db_retriever = vectorstore.as_retriever(search_kwargs={"k": top_n * 5})

def retrieve_then_rerank(question: str):
    """1차로 넉넉히 검색한 뒤, Cross-Encoder Reranker로 top_n개만 추립니다."""
    first_pass_docs = db_retriever.invoke(question)
    return reranker.compress_documents(first_pass_docs, question)

retrieval_chain = RunnableLambda(retrieve_then_rerank)

first_pass_docs = db_retriever.invoke(query)
final_docs = retrieval_chain.invoke(query)

show_results(first_pass_docs, f"1차 검색 (k={top_n * 5}개)")
show_results(final_docs, f"retrieval_chain 결과 (rerank 후 top {top_n}개)")

print(f"1차 검색에서 정답(p{ANSWER_PAGE}) 순위: {find_rank(first_pass_docs, ANSWER_PAGE)}위 / {len(first_pass_docs)}개")
print(f"retrieval_chain 이후 정답(p{ANSWER_PAGE}) 순위: {find_rank(final_docs, ANSWER_PAGE)}위 / {len(final_docs)}개")

--- 1차 검색 (k=20개) ---
[1순위][p27] 키오스크(무인정보단말기)이용 실태조사 28 시장조사국 시장감시팀 □(중단 이유)키오스크 이용을 도중...
[2순위][p22] 키오스크(무인정보단말기)이용 실태조사 23 시장조사국 시장감시팀 □(이용 이유)키오스크를 이용하게...
[3순위][p36] 키오스크(무인정보단말기)이용 실태조사 37 시장조사국 시장감시팀 -60대 이상은 다른 연령대에 비해...
[4순위][p4] 키오스크(무인정보단말기)이용 실태조사 5 시장조사국 시장감시팀 Ⅰ조사 개요1. 조사 배경 및 목적□...
[5순위][p51] 키오스크(무인정보단말기)이용 실태조사 52 시장조사국 시장감시팀 ㅇ 그러나 「장차법」시행일인’23년...
[6순위][p11] 키오스크(무인정보단말기)이용 실태조사 12 시장조사국 시장감시팀 [표2-2-3] 청년층·비장애인의...
[7순위][p34] 키오스크(무인정보단말기)이용 실태조사 35 시장조사국 시장감시팀 ㅇ (③큰 화면‧ 글씨,음성 안내...
[8순위][p6] 키오스크(무인정보단말기)이용 실태조사 7 시장조사국 시장감시팀 Ⅱ일반 현황1. 키오스크(무인정보단말...
[9순위][p25] 키오스크(무인정보단말기)이용 실태조사 26 시장조사국 시장감시팀 2. 키오스크 이용 불편·피해 실태...
[10순위][p8] 키오스크(무인정보단말기)이용 실태조사 9 시장조사국 시장감시팀 □(시장규모)세계 키오스크 시장은 ’...
[11순위][p19] 키오스크(무인정보단말기)이용 실태조사 20 시장조사국 시장감시팀 □(불만·피해 발생 원인)소비자 불...
[12순위][p2] 키오스크(무인정보단말기)이용 실태조사‘표’ 목차[표2-1-1] 키오스크의 종류 · · · · · ·...
[13순위][p21] 키오스크(무인정보단말기)이용 실태조사 22 시장조사국 시장감시팀 Ⅴ소비자 설문조사▣ 설문대상: 키오...
[14순위][p47] 키오스크(무인정보단말기)이용 실태조사 48 시장조사국 시장감시팀 Ⅶ문제점 및 개선방안1. 키오스크의...


`retrieval_chain`은 이제 질문 하나만 넣으면 "넉넉한 1차 검색 → Cross-Encoder rerank"까지 한 번에 처리하는 **하나의 retriever처럼** 동작합니다. 이 체인을 그대로 LCEL 프롬프트 체인이나 LangGraph 노드의 검색 단계에 끼워 넣으면, "1차 검색(top_n×5개) → rerank(top_n개) → 답변 생성"으로 이어지는 챗봇 그래프를 만들 수 있습니다.